# ParMetaD Free Energy Surfaces

Refined / restructured version of `ParMetaD_FES.ipynb`.

**Sections**
1. Setup (imports & constants)
2. Dataset 1 — `weights_master.dat` (1.72 μs, 1D RMSD)
3. Dataset 2 — `weights_master_2.dat` (1.74 μs, RMSD + Rg)
   - Exploration histograms
   - 1D FES (RMSD)
   - 2D FES (RMSD vs Rg)

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

In [ ]:
# --- Physical constants & analysis parameters ---
T          = 300.0                # Temperature (K)
kb_kJ      = 0.008314462618       # Boltzmann constant (kJ/mol/K)
kb_kcal    = 0.0019872            # Boltzmann constant (kcal/mol/K)
kT_kJ      = kb_kJ * T
kT_kcal    = kb_kcal * T
kJ_to_kcal = 0.239005736

BINS_1D = 100
BINS_2D = 100

In [ ]:
def compute_weights(rbias, we_weight, kT):
    """Combined WESTPA x MetaD reweighting weights, shifted to avoid exp overflow."""
    shift = np.max(rbias)
    return we_weight * np.exp((rbias - shift) / kT)

def fes_1d(cv, rbias, we_weight, kT, bins=BINS_1D, bin_edges=None):
    """Reweighted 1D FES. Returns (bin_centers, fes) with min(fes)=0."""
    w = compute_weights(rbias, we_weight, kT)
    hist, edges = np.histogram(cv, bins=bins if bin_edges is None else bin_edges, weights=w)
    fes = -kT * np.log(hist + 1e-12)
    fes -= np.min(fes)
    centers = 0.5 * (edges[:-1] + edges[1:])
    return centers, fes

def fes_2d(x, y, rbias, we_weight, kT, bins=BINS_2D, ranges=None, cap=None):
    """Reweighted 2D FES. Returns (X, Y, fes) with min(fes)=0; bins above cap become NaN."""
    w = compute_weights(rbias, we_weight, kT)
    hist, xe, ye = np.histogram2d(x, y, bins=bins, weights=w, range=ranges)
    hist = hist.T
    masked = np.ma.masked_where(hist <= 0, hist)
    fes = -kT * np.log(masked)
    fes -= np.min(fes)
    fes = fes.filled(np.nan)
    if cap is not None:
        fes[fes > cap] = np.nan
    xc = 0.5 * (xe[:-1] + xe[1:])
    yc = 0.5 * (ye[:-1] + ye[1:])
    X, Y = np.meshgrid(xc, yc)
    return X, Y, fes

## 2. Dataset 1 — `weights_master.dat` (1.72 μs)
Columns: `[cv, rbias, we_weight]`

In [ ]:
print('Loading weights_master.dat ...')
data = np.loadtxt('weights_master.dat')
cv        = data[:, 0]
rbias     = data[:, 1]
we_weight = data[:, 2]
print(f'  loaded {cv.size} frames')

### 2.1 1D FES

In [ ]:
centers, fes = fes_1d(cv, rbias, we_weight, kT_kJ)

# Save the curve for downstream use
np.savetxt('final_1D_FES.dat',
           np.column_stack((centers, fes)),
           header='RMSD(nm) Free_Energy(kJ/mol)', fmt='%.6f')
print('Saved final_1D_FES.dat')

In [ ]:
# FES with dual kJ/mol + kcal/mol axes
fig, ax1 = plt.subplots()
ax1.plot(centers * 10, fes, 'b')
ax1.set_xlabel(r'RMSD ($\AA$)')
ax1.set_ylabel('Free Energy (kJ/mol)', color='b')
ax1.set_ylim(0, 35)

ax2 = ax1.twinx()
ax2.set_ylim(0, 40 * kJ_to_kcal)
ax2.set_ylabel('Free Energy (kcal/mol)', color='r')

plt.title(r'ParMetaD (1.72 $\mu$s)')
plt.savefig('Chignolin_FES_both.png', dpi=300, bbox_inches='tight')
plt.show()

### 2.2 Unweighted RMSD histogram (exploration)

In [ ]:
weights_norm = np.ones_like(cv) / cv.size
plt.hist(cv * 10, bins=50, edgecolor='black', linewidth=1.2, density=True, weights=weights_norm)
plt.xlabel(r'RMSD ($\AA$)')
plt.ylabel('Fraction')
plt.title('Exploration \u2014 RMSD Histogram')
plt.tight_layout()
plt.savefig('cv_histogram.png', dpi=300)
plt.show()

## 3. Dataset 2 — `weights_master_2.dat` (1.74 μs)
Columns: `[cv, rg, rbias, we_weight]`

In [ ]:
print('Loading weights_master_2.dat ...')
data_2 = np.loadtxt('weights_master_2.dat')
cv_2        = data_2[:, 0]
rg_2        = data_2[:, 1]
rbias_2     = data_2[:, 2]
we_weight_2 = data_2[:, 3]
print(f'  loaded {cv_2.size} frames')
print(f'  first row: cv={cv_2[0]:.4f}  rg={rg_2[0]:.4f}  rbias={rbias_2[0]:.4f}  w={we_weight_2[0]:.4f}')

### 3.1 Exploration — unweighted 2D histograms

In [ ]:
# 2D histogram (log-count) of the first ~2.5M frames
n_preview = min(2_500_000, cv_2.size)

plt.hist2d(cv_2[:n_preview], rg_2[:n_preview], bins=1000, norm=LogNorm(vmin=1))
plt.colorbar(label='Counts')
plt.xlabel('RMSD (nm)')
plt.ylabel('Rg (nm)')
plt.title('Exploration \u2014 2D histogram (log counts)')
plt.tight_layout()
plt.show()

In [ ]:
# Unweighted visitation probability
H, xedges, yedges = np.histogram2d(cv_2, rg_2, bins=50)
P = H / H.sum()

plt.imshow(P.T, origin='lower',
           extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]],
           aspect='auto')
plt.colorbar(label='Visitation probability')
plt.xlabel('RMSD (nm)')
plt.ylabel('Rg (nm)')
plt.title('RMSD\u2013Rg exploration (unweighted)')
plt.tight_layout()
plt.show()

### 3.2 1D FES (RMSD)

In [ ]:
centers_2, fes_2 = fes_1d(cv_2, rbias_2, we_weight_2, kT_kJ)
print(f'  FES range: 0 – {np.max(fes_2):.2f} kJ/mol')

fig, ax1 = plt.subplots()
ax1.plot(centers_2 * 10, fes_2, 'b')
ax1.set_xlabel(r'RMSD ($\AA$)')
ax1.set_ylabel('Free Energy (kJ/mol)', color='b')
ax1.set_ylim(0, 35)

ax2 = ax1.twinx()
ax2.set_ylim(0, 40 * kJ_to_kcal)
ax2.set_ylabel('Free Energy (kcal/mol)', color='r')

plt.title(r'ParMetaD \u2014 RG-WESTPA (1.74 $\mu$s)')
plt.tight_layout()
plt.show()

In [ ]:
# Unweighted RMSD histogram for dataset 2
weights_norm_2 = np.ones_like(cv_2) / cv_2.size
plt.hist(cv_2 * 10, bins=50, edgecolor='black', linewidth=1.2, density=True, weights=weights_norm_2)
plt.xlabel(r'RMSD ($\AA$)')
plt.ylabel('Fraction')
plt.title('Exploration \u2014 RMSD Histogram (dataset 2)')
plt.tight_layout()
plt.show()

### 3.3 2D FES (RMSD vs Rg, kcal/mol)

In [ ]:
max_energy = 8.0   # cap for color scale (kcal/mol)

# Compute in Angstroms; rbias still in kJ/mol, so convert via kT_kcal
cv_ang = cv_2 * 10.0
rg_ang = rg_2 * 10.0
rbias_kcal = rbias_2 / 4.184

X, Y, fes2d = fes_2d(
    cv_ang, rg_ang, rbias_kcal, we_weight_2, kT_kcal,
    bins=BINS_2D, cap=max_energy,
)

In [ ]:
# Plot styling to match the original figure
plt.rcParams['axes.linewidth']    = 1.5
plt.rcParams['xtick.major.width'] = 1.5
plt.rcParams['ytick.major.width'] = 1.5

fig, ax = plt.subplots(figsize=(6, 5))

levels      = np.linspace(0, max_energy, 40)
tick_levels = np.linspace(0, max_energy, 10)

cf = ax.contourf(X, Y, fes2d, levels=levels, cmap='jet', extend='max')
ax.contour(X, Y, fes2d, levels=tick_levels, colors='black', linewidths=0.7, alpha=0.8)

ax.set_xlim(0, 10)
ax.set_ylim(3, 11)
ax.set_xlabel(r'RMSD ($\AA$)', fontsize=12)
ax.set_ylabel(r'Radius of Gyration ($\AA$)', fontsize=12)
ax.set_title(r'ParMetaD (RG-WESTPA) \u2014 1.74 $\mu$s', fontsize=14)

cbar = plt.colorbar(cf, ax=ax, ticks=tick_levels)
cbar.set_label('Free Energy (kcal/mol)', fontsize=12)
cbar.ax.set_yticklabels([f'{v:.1f}' for v in tick_levels])

plt.savefig('ParMetaD_FES_RG_2D.png', dpi=300, bbox_inches='tight')
plt.show()